In [5]:
import polars as pl
df_255 = pl.read_parquet(
    "datasets/dataset_merged_with_families.parquet",
)


In [6]:
import re
# Add sub_sequence column (21 aa from 'position')


WINDOW = 255
HALF = WINDOW // 2  # 256


def get_fragments_from_struct(row_dict: dict) -> dict:
    """
    Funkce extrahuje fragmenty o velikosti WINDOW.
    Přijímá slovník a vrací slovník s novými hodnotami.
    """
    mut_type = row_dict["mut_type"]
    original_seq = row_dict["original_seq_full"]
    mutated_seq = row_dict["mutated_seq_full"]

    # Získání první číselné hodnoty ze sloupce 'mut_type'
    match = re.search(r'\d+', mut_type)
    if not match:
        return {"fragment_255_mut": None, "fragment_255_org": None}

    # Pozice je 1-based, pro index v Pythonu odečteme 1
    center_index = int(match.group(0)) - 1
    seq_len = len(original_seq)

    # Výpočet počátečního a koncového indexu
    start = center_index - HALF
    end = start + WINDOW

    # Ošetření okrajů sekvence
    if start < 0:
        start = 0
        end = WINDOW
    if end > seq_len:
        end = seq_len
        start = seq_len - WINDOW
    if start < 0: # Zajištění pro sekvence kratší než WINDOW
        start = 0

    # Extrakce fragmentů
    org_fragment = original_seq[start:end]
    mut_fragment = mutated_seq[start:end]

    # Funkce musí vrátit slovník s názvy budoucích sloupců
    return {"fragment_255_mut": mut_fragment, "fragment_255_org": org_fragment}

# Aplikace funkce na DataFrame
df_255 = df_255.with_columns(
    # 1. Seskupíme potřebné sloupce do dočasné struktury
    pl.struct(["mut_type", "original_seq_full", "mutated_seq_full"])
    # 2. Aplikujeme funkci na tuto strukturu.
    #    Je nutné specifikovat návratový typ pro optimalizaci.
    .map_elements(
        get_fragments_from_struct,
        return_dtype=pl.Struct([
            pl.Field("fragment_255_mut", pl.String),
            pl.Field("fragment_255_org", pl.String),
        ])
    )
    # 3. Dáme výsledné struktuře dočasný název
    .alias("fragments_struct")
).unnest("fragments_struct") # 4. Rozbalíme strukturu do finálních sloupců



df_255

original_seq_full,mutated_seq_full,mut_type,target,reverse,data_source,cath_dominant,cath_all,cath_class,cath_arch,cath_topology,cath_homology,fragment_255_mut,fragment_255_org
str,str,str,f64,bool,str,str,str,str,str,str,str,str,str
"""SAGGSAGGSAGGSNSLAEAKVLANRELDKY…","""SAGGSAGGSAGGSNSLAEAKVLANRELDKY…","""I27T""",-0.352598,false,"""megascale""","""G3DSA:1.20.900.10""","""G3DSA:1.10.238.10;G3DSA:1.20.9…","""G3DSA:1""","""20""","""900""","""10""","""SAGGSAGGSAGGSNSLAEAKVLANRELDKY…","""SAGGSAGGSAGGSNSLAEAKVLANRELDKY…"
"""SAGGSAEIMKKTDFDKVASEYTKIGTISTT…","""SAGGSAEIMKKTDFDKVASEYTLIGTISTT…","""K18L:D65P""",-0.157241,false,"""megascale""","""G3DSA:1.10.287.540""","""G3DSA:1.10.287.540""","""G3DSA:1""","""10""","""287""","""540""","""SAGGSAEIMKKTDFDKVASEYTLIGTISTT…","""SAGGSAEIMKKTDFDKVASEYTKIGTISTT…"
"""SAGGSEVTIKANLIFANGSTQTAEFKGTFE…","""SAGGSEVTIKANLIFANGSGQTAEFKGTFE…","""T15G""",-0.166278,false,"""megascale""","""G3DSA:1.10.10.10""","""G3DSA:1.10.10.10;G3DSA:1.10.15…","""G3DSA:1""","""10""","""10""","""10""","""SAGGSEVTIKANLIFANGSGQTAEFKGTFE…","""SAGGSEVTIKANLIFANGSTQTAEFKGTFE…"
"""SAGGSAVTTYKLVINGKTLKGETTTKAVDA…","""SAGGSAVTTYKRVINGKTLKGETTTKAVDA…","""L6R""",-0.580267,false,"""megascale""","""G3DSA:1.20.1270.60""","""G3DSA:1.20.1270.60;G3DSA:2.30.…","""G3DSA:1""","""20""","""1270""","""60""","""SAGGSAVTTYKRVINGKTLKGETTTKAVDA…","""SAGGSAVTTYKLVINGKTLKGETTTKAVDA…"
"""SAGGSAGGSAGGTTYKLILNGKTLKGETTT…","""SAGGSAGGSAGGTTYKHILNGKTLKGETTT…","""L5H:F30N""",-0.675053,false,"""megascale""","""G3DSA:1.10.10.60""","""G3DSA:1.10.10.60""","""G3DSA:1""","""10""","""10""","""60""","""SAGGSAGGSAGGTTYKHILNGKTLKGETTT…","""SAGGSAGGSAGGTTYKLILNGKTLKGETTT…"
…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""MSNVSEERRKRQQNIKEGLQFIQSPLSYPG…","""MSNVSEERRKRQQNIKEGLQFIQSPLSYPG…","""S906I""",-0.140346,false,"""lehner""","""G3DSA:1.10.10.60""","""G3DSA:1.10.10.60""","""G3DSA:1""","""10""","""10""","""60""","""RNGQVIEPDKNRKYCSAKARHSWTKDRRAM…","""RNGQVIEPDKNRKYCSAKARHSWTKDRRAM…"
"""MSNVSEERRKRQQNIKEGLQFIQSPLSYPG…","""MSNVSEERRKRQQNIKEGLQFIQSPLSYPG…","""T906I""",-0.183833,false,"""lehner""","""G3DSA:1.10.10.60""","""G3DSA:1.10.10.60""","""G3DSA:1""","""10""","""10""","""60""","""RNGQVIEPDKNRKYCSAKARHSWTKDRRAM…","""RNGQVIEPDKNRKYCSAKARHSWTKDRRAM…"
"""MSNVSEERRKRQQNIKEGLQFIQSPLSYPG…","""MSNVSEERRKRQQNIKEGLQFIQSPLSYPG…","""V906I""",-0.11009,false,"""lehner""","""G3DSA:1.10.10.60""","""G3DSA:1.10.10.60""","""G3DSA:1""","""10""","""10""","""60""","""RNGQVIEPDKNRKYCSAKARHSWTKDRRAM…","""RNGQVIEPDKNRKYCSAKARHSWTKDRRAM…"


In [8]:
import polars as pl
import random

# Předpokládáme, že df_255 je již načtený
# Důležité: Ujistěte se, že df_255 obsahuje sloupec "cath_homology"

TRAIN_RATIO = 0.8
VAL_RATIO = 0.1
TEST_RATIO = 0.1

# 0. Získání celkového počtu řádků
n_rows = df_255.height
print(f"Celkový počet řádků v datasetu: {n_rows}")

# -------------------------------------------------------------------------
# KROK 1: Analýza rodin (Homology)
# -------------------------------------------------------------------------
# Spočítáme, kolik řádků má každá rodina
family_counts = (
    df_255
    .group_by("cath_homology")
    .len()
    .rename({"len": "count"})
)

# Převedeme na list slovníků pro iteraci v Pythonu (je to rychlejší pro logiku "bin packing")
# Výsledek: [{'cath_homology': '1.10.10.10', 'count': 500}, ...]
families_list = family_counts.to_dicts()

# DŮLEŽITÉ: Náhodně zamícháme rodiny.
# Pokud bychom je seřadili podle velikosti, mohli bychom do testu dát jen ty obří
# a model by se učil jen na malých (nebo naopak). Chceme reprezentativní mix.
random.seed(42) # Fixní seed pro reprodukovatelnost
random.shuffle(families_list)

# -------------------------------------------------------------------------
# KROK 2: Rozdělování rodin do skupin (Bin Packing)
# -------------------------------------------------------------------------
test_families = []
val_families = []
train_families = []

current_test_count = 0
current_val_count = 0

target_test_count = int(n_rows * TEST_RATIO)
target_val_count = int(n_rows * VAL_RATIO)

# Iterujeme přes zamíchané rodiny a plníme košíky
for fam in families_list:
    fam_id = fam["cath_homology"]
    count = fam["count"]

    # 1. Naplníme nejdříve Testovací sadu
    if current_test_count < target_test_count:
        test_families.append(fam_id)
        current_test_count += count

    # 2. Pak naplníme Validační sadu
    elif current_val_count < target_val_count:
        val_families.append(fam_id)
        current_val_count += count

    # 3. Zbytek jde do Trénovací sady
    else:
        train_families.append(fam_id)

print(f"\nRozdělení rodin:")
print(f"  Test:  {len(test_families)} rodin (cca {current_test_count} řádků)")
print(f"  Val:   {len(val_families)} rodin (cca {current_val_count} řádků)")
print(f"  Train: {len(train_families)} rodin (zbytek)")

# -------------------------------------------------------------------------
# KROK 3: Vytvoření Datasetů filtrováním
# -------------------------------------------------------------------------
# Použijeme is_in() pro rychlé vyfiltrování

test_df = df_255.filter(pl.col("cath_homology").is_in(test_families))
val_df = df_255.filter(pl.col("cath_homology").is_in(val_families))
train_df = df_255.filter(pl.col("cath_homology").is_in(train_families))

# (Volitelné) Vytvoření testovacího setu bez reverzních mutací
# Předpokládám, že sloupec "reverse" existuje
if "reverse" in test_df.columns:
    test_df_noreverse = test_df.filter(pl.col("reverse") == False)
else:
    test_df_noreverse = test_df # Pokud sloupec není, je to stejné
    print("Warning: Sloupec 'reverse' nenalezen, noreverse set je shodný s test setem.")


# -------------------------------------------------------------------------
# KROK 4: Uložení
# -------------------------------------------------------------------------

# Definice prefixu pro názvy souborů
DATASET_NAME_PERFIX = "datasets/dataset_255w_homology_split_"

# Uložení (Parquet je lepší, ale nechávám CSV dle vašeho původního kódu)
train_df.write_csv(f"{DATASET_NAME_PERFIX}train.csv")
val_df.write_csv(f"{DATASET_NAME_PERFIX}validation.csv")
test_df.write_csv(f"{DATASET_NAME_PERFIX}test.csv")
test_df_noreverse.write_csv(f"{DATASET_NAME_PERFIX}_noreverse_test.csv")

# -------------------------------------------------------------------------
# KROK 5: Výpis finálních statistik
# -------------------------------------------------------------------------
real_train_len = len(train_df)
real_val_len = len(val_df)
real_test_len = len(test_df)
total_len = real_train_len + real_val_len + real_test_len

print("\n--- Výsledky finálního rozdělení (podle Homologie) ---")
print(f"Trénovací sada:   {real_train_len:>8} řádků ({real_train_len/total_len:>6.1%})")
print(f"Validační sada:    {real_val_len:>8} řádků ({real_val_len/total_len:>6.1%})")
print(f"Testovací sada:     {real_test_len:>8} řádků ({real_test_len/total_len:>6.1%})")
print("-------------------------------")
print(f"Celkem zpracováno: {total_len:>8} řádků")

# Kontrola průniku (musí být 0)
# Ověříme, že se žádná rodina nenachází ve více sadách
intersection_check = set(test_families).intersection(set(train_families))
if len(intersection_check) == 0:
    print("\n✅ KONTROLA OK: Žádná rodina z Test setu není v Train setu.")
else:
    print(f"\n❌ CHYBA: Průnik rodin nalezen! {intersection_check}")

print(f"\nSoubory byly úspěšně uloženy s prefixem '{DATASET_NAME_PERFIX}'.")

Celkový počet řádků v datasetu: 1738861

Rozdělení rodin:
  Test:  6 rodin (cca 181329 řádků)
  Val:   7 rodin (cca 217975 řádků)
  Train: 45 rodin (zbytek)

--- Výsledky finálního rozdělení (podle Homologie) ---
Trénovací sada:    1339557 řádků ( 77.0%)
Validační sada:      217975 řádků ( 12.5%)
Testovací sada:       181329 řádků ( 10.4%)
-------------------------------
Celkem zpracováno:  1738861 řádků

✅ KONTROLA OK: Žádná rodina z Test setu není v Train setu.

Soubory byly úspěšně uloženy s prefixem 'datasets/dataset_255w_homology_split_'.


In [4]:
train_df

original_seq_full,mutated_seq_full,mut_type,target,reverse,data_source,fragment_255_mut,fragment_255_org
str,str,str,f64,bool,str,str,str
"""SAGGSAGGSAGGHEITLHINGRRVKLRFRD…","""SAGGSAGGSAGGHEITLHINGRRVKLRFTD…","""R17T""",-0.110833,false,"""megascale""","""SAGGSAGGSAGGHEITLHINGRRVKLRFTD…","""SAGGSAGGSAGGHEITLHINGRRVKLRFRD…"
"""SAGGNKASVVANQLIPINTALTLIMMKAEV…","""SAGGNKASVVANQLIPINTALTLIMMKAEV…","""K39N""",-0.038819,false,"""megascale""","""SAGGNKASVVANQLIPINTALTLIMMKAEV…","""SAGGNKASVVANQLIPINTALTLIMMKAEV…"
"""SENVVSAPMPGKVLRVLVRVGDRVRVGQGL…","""SENVVSAPMPGKVLRVLVRVGDRVRVPQGL…","""G26P""",-0.384118,false,"""megascale""","""SENVVSAPMPGKVLRVLVRVGDRVRVPQGL…","""SENVVSAPMPGKVLRVLVRVGDRVRVGQGL…"
"""LQLFIKTLTGKTFTVEMEPSDTIENLKAKI…","""LQLFIKTLTGKTFTVEMEPSDTIENLKAKI…","""Q49L""",-0.36507,false,"""megascale""","""LQLFIKTLTGKTFTVEMEPSDTIENLKAKI…","""LQLFIKTLTGKTFTVEMEPSDTIENLKAKI…"
"""SAGGSAGGSAGGSNSLAEAKVLANRELDKY…","""SAGGSAGGSAGGSNSLAEAKVLANRELDKY…","""I27T""",-0.352598,false,"""megascale""","""SAGGSAGGSAGGSNSLAEAKVLANRELDKY…","""SAGGSAGGSAGGSNSLAEAKVLANRELDKY…"
…,…,…,…,…,…,…,…
"""MGNEASLEGEGLPEGLAAAAAAGGGASGAG…","""MGNEASLEGEGLPEGLAAAAAAGGGASGAG…","""V1059C""",-0.45026,false,"""lehner""","""FGFGASIFSQASNLISTAGQPGPHSQSGPG…","""FGFGASIFSQASNLISTAGQPGPHSQSGPG…"
"""MGNEASLEGEGLPEGLAAAAAAGGGASGAG…","""MGNEASLEGEGLPEGLAAAAAAGGGASGAG…","""W1059C""",-0.374472,false,"""lehner""","""FGFGASIFSQASNLISTAGQPGPHSQSGPG…","""FGFGASIFSQASNLISTAGQPGPHSQSGPG…"
"""MGNEASLEGEGLPEGLAAAAAAGGGASGAG…","""MGNEASLEGEGLPEGLAAAAAAGGGASGAG…","""Y1059C""",-0.462593,false,"""lehner""","""FGFGASIFSQASNLISTAGQPGPHSQSGPG…","""FGFGASIFSQASNLISTAGQPGPHSQSGPG…"


In [11]:
df_255.filter(pl.col("original_seq_full").is_in(holdout_sequences))

original_seq_full,mutated_seq_full,mut_type,target,reverse,data_source,fragment_255_mut,fragment_255_org
str,str,str,f64,bool,str,str,str
"""SAGGSIIYNLKLIREKKKISQSELAALLEV…","""SAGGSIINNLKLIREKKKISQSELAALLEV…","""N3Y""",0.313876,true,"""megascale""","""SAGGSIINNLKLIREKKKISQSELAALLEV…","""SAGGSIIYNLKLIREKKKISQSELAALLEV…"
"""SAGGSAGGSAGGKELVLVLYDYQEKSPREV…","""SAGGSAGGSAGGKELVLVLYDYQEKSPREV…","""K21C""",0.015784,true,"""megascale""","""SAGGSAGGSAGGKELVLVLYDYQEKSPREV…","""SAGGSAGGSAGGKELVLVLYDYQEKSPREV…"
"""SAGGMNLTVNGKPSTVDGAESLNVTELLSA…","""SAGGMNLTVNGKPSTVDGAESLNVTELLSA…","""R46T""",0.190525,true,"""megascale""","""SAGGMNLTVNGKPSTVDGAESLNVTELLSA…","""SAGGMNLTVNGKPSTVDGAESLNVTELLSA…"
"""SAGGSAGGSAGGDEVRLHVNGHTGEFRGID…","""SAGGSAGGSAGGDEVRLHVNGHTIEFRGID…","""I12G""",0.147245,true,"""megascale""","""SAGGSAGGSAGGDEVRLHVNGHTIEFRGID…","""SAGGSAGGSAGGDEVRLHVNGHTGEFRGID…"
"""SAGGSAGGSAGGTYYTVKSGDTANKIAAQY…","""SAGGSAGGSAGGTYYTVKSGDTANKIAAQY…","""V20E""",0.210962,true,"""megascale""","""SAGGSAGGSAGGTYYTVKSGDTANKIAAQY…","""SAGGSAGGSAGGTYYTVKSGDTANKIAAQY…"
…,…,…,…,…,…,…,…
"""MDPGAGSETSLTVNEQVIVMSGHETIRVLE…","""MDPGAGSETSLTVNEQVIVMSGHETIRVLE…","""S202K""",-0.359692,false,"""lehner""","""GSSAEATVKSPPGIPPSPATAIATFSQAPS…","""GSSAEATVKSPPGIPPSPATAIATFSQAPS…"
"""MDPGAGSETSLTVNEQVIVMSGHETIRVLE…","""MDPGAGSETSLTVNEQVIVMSGHETIRVLE…","""T202K""",-0.13698,false,"""lehner""","""GSSAEATVKSPPGIPPSPATAIATFSQAPS…","""GSSAEATVKSPPGIPPSPATAIATFSQAPS…"
"""MDPGAGSETSLTVNEQVIVMSGHETIRVLE…","""MDPGAGSETSLTVNEQVIVMSGHETIRVLE…","""V202K""",-0.027627,false,"""lehner""","""GSSAEATVKSPPGIPPSPATAIATFSQAPS…","""GSSAEATVKSPPGIPPSPATAIATFSQAPS…"


In [12]:
holdout_sequences

['...*DIIKILNKKGQQGWWRGEIYGRVGWFPANYVEEDYSEYC',
 '...']